# Experimento: FT-Transformer, fase 7 (pacotes, target encoding e Optuna com os dados)

**De onde partimos.** O melhor FT é o `o5_top3` (placar 0,5562, KS OOF 0,5508). Duas evidências apontam para onde ir:

- **Fase 6.** Nenhuma feature passou sozinha na seleção, mas cinco deram ganho no KS OOF: `cobranca_razao` (+0,0047, positivo em 75% das reamostragens), `antigo`, `caro_sem_backup`, `fibra_sem_protecao` e `atrito_pagamento` (entre +0,0024 e +0,0031). Juntas, podem somar.
- **Curva de aprendizado.** Com 25%, 50%, 75% e 100% do treino, o KS sobe até 75% e fica plano. A perda no treino (0,409) é quase igual à da validação (0,413). A rede não está decorando: o que falta é **informação nova na entrada**.

## 1. Triagem nas divisões do Optuna

Cada candidato é medido sobre o `o5_top3` nas **divisões próprias do Optuna** (sementes 201 a 203, 15 dobras), nunca nas do placar. As medidas da base e das features da fase 6 vêm do cache da fase 6 e não se repetem.

| Candidato | O que acrescenta |
|---|---|
| `p_razao_antigo` | `cobranca_razao` + `antigo` |
| `p_cinco` | as cinco com ganho no OOF da fase 6 |
| `te_contrato_pag_net` | taxa de churn do grupo contrato × forma de pagamento × internet |
| `te_tenure_contrato` | taxa de churn do grupo faixa de tempo de casa (≤1, ≤6, ≤12, ≤24, ≤48, mais) × contrato |
| `te_net_protecao` | taxa de churn do grupo internet × suporte técnico × segurança × backup |
| `te_preco_contrato` | taxa de churn do grupo faixa de mensalidade (≤35, ≤70, ≤90, ≤100, mais) × contrato |
| `te_todos` | os quatro target encodings |
| `p_cinco_te_todos` | o pacote de cinco e os quatro target encodings |

**Target encoding sem vazamento** (`comum.codificar_alvo`). A taxa é calculada **só com o treino da dobra**. Os clientes de avaliação recebem a taxa do treino inteiro da dobra. Cada cliente do treino recebe a taxa calculada nas outras 4 partes do treino, então o próprio rótulo nunca entra na sua codificação. A taxa é suavizada com 20 clientes "virtuais" na média geral, para grupos pequenos não ficarem extremos. As faixas são fixas, escolhidas antes de olhar o rótulo.

## 2. Optuna com o conjunto de dados como eixo

O Optuna escolhe entre a base e os dois melhores candidatos da triagem, junto com a arquitetura e a regularização. A região de busca fica em volta do que a fase 5 achou: 1 ou 2 blocos, `d_token` de 128 a 256, dropout de atenção de 0,2 a 0,6, sem oversampling. Os primeiros trials são o `o5_top3` com cada conjunto. O estudo fica em sqlite na pasta da execução: retomar a execução retoma o estudo.

## 3. Placar

Vão ao placar: o melhor candidato da triagem com os hiperparâmetros do `o5_top3` (`f7_triagem`), os 3 melhores trials (`o7_top1` a `o7_top3`) e o melhor com a média de 3 redes. Os vereditos são contra o `o5_top3`, contra o melhor FT (`o5_top1_3redes`), contra o trial 0 e contra o MLP adotado.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase7.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*. São cerca de 6 h. Para continuar uma execução que caiu, use `RUN_NAME` com o mesmo nome e `RESUME_FROM = "github"`: as medidas e os trials já feitos não se repetem.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase7-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE6_RUN = "ft-fase6-20261119-1518"   # execução da fase 6 na branch de resultados (ou uma pasta local): traz o placar e o cache da seleção
N_TRIALS = 60                   # trials do Optuna (cada um mede 15 dobras próprias)
TOP_OPTUNA = 3                  # melhores trials que vão para o placar
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase7-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    N_TRIALS, TOP_OPTUNA = 3, 1
    print("MODO_TESTE: 3 épocas, dois candidatos e 3 trials. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase4 as F4, fase5 as F5, fase6 as F6, fase7 as F7

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=7, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 a 6 (com o trial 0, o o5_top3 e o o5_top1_3redes) e o cache da seleção da fase 6 entram aqui sem treinar de novo
import tempfile
if FASE6_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE6_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE6_RUN) else \
            fluxo.baixar_github(_tmp, FASE6_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
        if ok and os.path.isfile(os.path.join(_tmp, "selecao.jsonl")) and not MODO_TESTE:   # medidas da fase 6 nas divisões do Optuna
            shutil.copy(os.path.join(_tmp, "selecao.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "selecao_oof"), os.path.join(PASTA, "selecao_oof"), dirs_exist_ok=True)
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 a 6: {len(anteriores)} linhas trazidas de {FASE6_RUN}" if anteriores else "Fases 1 a 6: nada trazido; as bases serão medidas aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")
Xf_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5], features=F4.TRES)
print(f"com as features do MLP: {Xf_.shape[1]} tokens + [CLS] ({', '.join(F4.TRES)})")
for nome, hp in (("tokens categóricos", dict(tokens="cat")), ("PLR", dict(num_emb="plr"))):
    g_, n_ = M.estrutura(hp, list(Xf_.columns), np.asarray(Xf_, dtype="float32"))
    print(f"{nome}: {len(g_)} tokens | colunas com PLR: {[Xf_.columns[j] for j in n_]}")
# Target encoding: grupos, e a garantia de que o próprio rótulo não entra na codificação do cliente
y_ = np.asarray(ytr_)
for t in F7.TES:
    g = comum.TE_GRUPOS[t](Xtr_)
    taxa = pd.Series(y_).groupby(g.to_numpy()).agg(["mean", "size"])
    print(f"{t:<22} {len(taxa):>3} grupos | churn por grupo de {taxa['mean'].min():.2f} a {taxa['mean'].max():.2f} | menor grupo {taxa['size'].min()} clientes")
A_, _ = comum.codificar_alvo(Xtr_, ytr_, Xtr_.iloc[:5], F7.TES)
y2_ = ytr_.copy(); y2_.iloc[0] = 1 - y2_.iloc[0]
A2_, _ = comum.codificar_alvo(Xtr_, y2_, Xtr_.iloc[:5], F7.TES)
assert all(A_[t].iloc[0] == A2_[t].iloc[0] for t in F7.TES), "o rótulo do cliente vazou na própria codificação"
print("target encoding: trocar o rótulo de um cliente não muda a codificação dele")

# 1. Triagem nas divisões do Optuna

Cada linha é um candidato somado ao `o5_top3`. `delta_oof` é o ganho no KS OOF contra a base, e `p_positivo` é a fração das reamostragens do bootstrap com ganho. `passou` = ganho em pelo menos 80% das reamostragens. Os dois de maior `delta_oof` (com ganho) entram no Optuna como opção de conjunto de dados.

In [ ]:
assert MODO_TESTE or (F7.ROT_BASE in anteriores and F1._chave(anteriores[F7.ROT_BASE]["config"]) == F1._chave(F7.BASE)), \
    "o o5_top3 trazido não bate com fase7.BASE"
CANDS = F7.candidatos()
if MODO_TESTE:
    CANDS = {k: CANDS[k] for k in ("p_razao_antigo", "te_tenure_contrato")}
D_OPT = F5.divs_optuna(F5.SEMENTES_OPTUNA[:1] if MODO_TESTE else F5.SEMENTES_OPTUNA)
y_tr = np.asarray(F1._dados()[2])


def ao_medir(l):
    print(f"  {l['candidato']:<22} KS {l['ks_kfold']:.4f} ({l['delta_kfold']:+.4f}) | delta OOF {l['delta_oof']:+.4f} p+ {l['p_positivo']:.2f}", flush=True)
    if wandb_run is not None:
        wandb_run.log({"triagem/delta_oof": l["delta_oof"], "triagem/p_positivo": l["p_positivo"]})
    salvar()


t0 = time.time()
linhas_t = F7.triagem(PASTA, lambda c: F1.medir(c, DISPOSITIVOS, POR_GPU, D=D_OPT), y_tr, CANDS, continuar=continuar, ao_medir=ao_medir)
print(f"{(time.time() - t0) / 60:.1f} min")
tt = pd.DataFrame(linhas_t).sort_values("delta_oof", ascending=False)
tt["ic95"] = tt.ic95.map(lambda v: f"[{v[0]:+.4f}, {v[1]:+.4f}]")
display(tt.round(4))
arq_conj = os.path.join(PASTA, "optuna_conjuntos.json")
if os.path.isfile(arq_conj):                       # retomar: o eixo do Optuna não pode mudar no meio do estudo
    CONJ = json.load(open(arq_conj))
else:
    CONJ = F7.conjuntos_optuna(linhas_t, CANDS)
    json.dump(CONJ, open(arq_conj, "w"), indent=1)
json.dump(linhas_t, open(os.path.join(PASTA, "triagem.json"), "w"), indent=1)
print("Conjuntos no Optuna:", list(CONJ))
_ = salvar()

# 2. Optuna (conjunto de dados, arquitetura e regularização)

In [ ]:
import optuna


def ao_trial(st, t, r):
    if wandb_run is not None:
        wandb_run.log({"optuna/trial": t.number, "optuna/ks": r["ks_kfold"], "optuna/melhor": st.best_value, "optuna/minutos": r.get("tempo_min")})
    salvar()


t0 = time.time()
estudo = F7.estudar(PASTA, N_TRIALS, CONJ, medir=lambda c, D: F1.medir(c, DISPOSITIVOS, POR_GPU, D=D),
                    continuar=continuar, ao_trial=ao_trial, D=D_OPT)
print(f"{(time.time() - t0) / 60:.1f} min | {len(estudo.trials)} trials")
df_opt = estudo.trials_dataframe().sort_values("value", ascending=False)
df_opt.to_csv(os.path.join(PASTA, "optuna_trials.csv"), index=False)
display(df_opt.head(10).round(4))
res_opt = F5.resumo_optuna(estudo)
res_opt.to_csv(os.path.join(PASTA, "optuna_resumo.csv"), index=False)
display(res_opt.round(4))
try:
    imp = optuna.importance.get_param_importances(estudo)
    print("Importância dos eixos (fANOVA):"); display(pd.Series(imp).round(3).to_frame("importância"))
    json.dump(imp, open(os.path.join(PASTA, "optuna_importancia.json"), "w"), indent=1)
except Exception as e:
    print("importância indisponível:", e)
_ = salvar()

# 3. No placar

`vs_base` é contra o `o5_top3`, `vs_melhor` contra o `o5_top1_3redes` (o maior KS do placar até agora). A última tabela compara com o MLP adotado.

In [ ]:
T0, B, MELHOR = ("ft_transformer", "trial0"), ("ft_transformer", F7.ROT_BASE), ("ft_transformer", "o5_top1_3redes")
contra = {"vs_trial0": T0, "vs_base": B, **({"vs_melhor": MELHOR} if "o5_top1_3redes" in anteriores else {})}
itens = [("trial0", F1.TRIAL0, None), (F7.ROT_BASE, F7.BASE, {"vs_trial0": T0})]
bons = sorted([l for l in linhas_t if l["delta_oof"] > 0], key=lambda l: -l["delta_oof"])
if bons:
    itens.append(("f7_triagem", CANDS[bons[0]["candidato"]], contra))
    print("f7_triagem =", bons[0]["candidato"])
tops = [t for t in F5.melhores(estudo, TOP_OPTUNA + len(CONJ))
        if F1._chave(F7.config_do_trial(t.params, CONJ)) not in {F1._chave(c) for _, c, _ in itens}][:TOP_OPTUNA]
itens_o = [(f"o7_top{i + 1}", F7.config_do_trial(t.params, CONJ), contra) for i, t in enumerate(tops)]
if itens_o:
    c1 = itens_o[0][1]
    itens_o.append(("o7_top1_3redes", dict(hp=dict(c1["hp"], n_redes=3), dados=json.loads(json.dumps(c1["dados"]))), contra))
for (r, c, _), t in zip(itens_o, tops + tops[:1]):
    print(r, f"| trial {t.number} | KS Optuna {t.value:.4f} |", json.dumps(c, sort_keys=True))
linhas7 = medir(itens + itens_o)
display(F1.tabela(linhas7, ("vs_trial0", "vs_base", "vs_melhor")).round(4))

RES = os.path.join(REPO_DIR, "resultados")
rot_mlp = json.load(open(os.path.join(RES, "escolhidos.json")))["mlp"]["rotulo"]
mlp = {r["rotulo"]: r for r in comum.ler_placar(RES) if r["modelo"] == "mlp"}[rot_mlp]
oof_mlp = np.load(os.path.join(RES, mlp["oof"]))
vv = lambda v: f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
novos = {"o5_top3", "o5_top1_3redes", "f7_triagem"} | {r for r, _, _ in itens_o}
comp = [dict(versao=r["rotulo"], ks_kfold=r["ks_kfold"], ks_oof=r["ks_oof"],
             vs_mlp=vv(comum.comparar_placar(y_tr, oof_mlp, np.load(os.path.join(PASTA, r["oof"])))))
        for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer" and r["rotulo"] in novos]
print(f"Contra o MLP adotado ({rot_mlp}, KS do placar {mlp['ks_kfold']:.4f})")
display(pd.DataFrame(comp).sort_values("ks_kfold", ascending=False).round(4))
_ = salvar()

# 4. Placar das fases 1 a 7

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base", "vs_melhor"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase7.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 7

- **Algum pacote ou target encoding ganhou na triagem?** _…_
- **Qual conjunto o Optuna preferiu, e com que arquitetura?** _…_
- **Algum `o7_*` ou o `f7_triagem` ganhou do `o5_top3` e do `o5_top1_3redes` no placar?** _…_
- **E contra o MLP?** _…_
- **Próximo passo** (o mesmo conjunto no MLP, adotar e medir o teste): _…_